# TOML - JavaScript

All 8 JavaScript examples from [docs/toml.md](https://platob.github.io/yggdryl/toml/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const { toml } = require('yggdryl')

const source = 'title = "yggdryl"\ncount = 3\n\n[owner]\nname = "Ada"\n'
const value = toml.loads(source)

assert.deepEqual(value, { title: 'yggdryl', count: 3, owner: { name: 'Ada' } })
assert.equal(value.owner.name, 'Ada')
assert.deepEqual(toml.loads(toml.dumps(value)), value)

## Table order

In [ ]:
const assert = require('node:assert/strict')
const { toml } = require('yggdryl')

const value = { zeta: 1, beta: 2, alpha: { deep: 3 } }

const encoded = toml.dumps(value)
assert.equal(
  encoded.toString('utf8'),
  '"zeta" = 1\n"beta" = 2\n"alpha" = {"deep" = 3}\n',
)

// Decoding rebuilds a plain object out of a sorted map.
assert.deepEqual(Object.keys(toml.loads(encoded)), ['alpha', 'beta', 'zeta'])

## The type mapping

In [ ]:
const assert = require('node:assert/strict')
const { Value, toml } = require('yggdryl')

const value = toml.loads(
  'text = "café"\n' +
    'integer = 7\n' +
    'hex = 0x2a\n' +
    'float = 1.5\n' +
    'infinite = inf\n' +
    'negative_zero = -0.0\n' +
    'flag = true\n' +
    'array = [1, "two"]\n' +
    'table = { nested = 1 }\n' +
    'moment = 1979-05-27T07:32:00Z\n',
)

assert.equal(typeof value.text, 'string')
assert.equal(value.integer, 7)
assert.equal(value.hex, 42)
assert.equal(value.float, 1.5)
assert.equal(value.infinite, Infinity)
assert.ok(Object.is(value.negative_zero, -0))
assert.equal(value.flag, true)
assert.deepEqual(value.array, [1, 'two'])
assert.deepEqual(value.table, { nested: 1 })
assert.ok(value.moment.equals(Value.timestamp(296638320n, 's', 'UTC')))

## Values TOML has no syntax for

In [ ]:
const assert = require('node:assert/strict')
const { toml } = require('yggdryl')

const value = {
  missing: null,
  blob: Buffer.from([0, 255]),
  seen: new Map([[1, 'one']]),
}

const encoded = toml.dumps(value).toString('utf8')
assert.ok(
  encoded.includes('"missing" = { "$yggdryl" = { version = 1, type = "null" } }'),
)

const decoded = toml.loads(encoded)
assert.equal(decoded.missing, null)
assert.deepEqual(decoded.blob, value.blob)
assert.ok(decoded.seen instanceof Map)
assert.equal(decoded.seen.get(1), 'one')

// A TOML root is a table, so a non-table root is wrapped the same way.
assert.equal(toml.loads(toml.dumps('scalar root')), 'scalar root')

// A user table that only looks like an envelope stays user data.
const lookalike = { $yggdryl: { version: 1, type: 'null' } }
assert.deepEqual(toml.loads(toml.dumps(lookalike)), lookalike)

## Dates and times

In [ ]:
const assert = require('node:assert/strict')
const { Value, toml } = require('yggdryl')

const value = toml.loads(
  'offset = 1979-05-27T07:32:00Z\n' +
    'local = 1979-05-27T07:32:00\n' +
    'day = 1979-05-27\n' +
    'clock = 07:32:00\n',
)

// A Date is a naive count of milliseconds, which is what a local
// date-time is, so that one form arrives as a Date.
assert.ok(value.local instanceof Date)
assert.equal(value.local.toISOString(), '1979-05-27T07:32:00.000Z')

// The other three have no JavaScript object of their own, so they stay
// native values rather than being rounded into a Date.
assert.ok(value.offset.equals(Value.timestamp(296638320n, 's', 'UTC')))
assert.ok(value.day.equals(Value.date(3433)))
assert.ok(value.clock.equals(Value.time(27120n, 's')))

// Each form goes back out in the syntax it arrived in.
assert.match(toml.dumps(value).toString('utf8'), /"day" = 1979-05-27\n/)

## Exactly one document

In [ ]:
const assert = require('node:assert/strict')
const { toml } = require('yggdryl')

// The root is a table, so an empty or comment-only document is an empty table.
assert.deepEqual(toml.loads('# nothing to see\n'), {})
assert.equal(toml.dumps({}).length, 0)

// There is no multi-document pair, only the single-document one.
assert.equal(toml.loadsAll, undefined)
assert.equal(toml.dumpAll, undefined)

## Failures

In [ ]:
const assert = require('node:assert/strict')
const { toml } = require('yggdryl')

assert.throws(() => toml.loads('ok = 0\nnested = { a = 1, a = 2 }\n'), /duplicate/i)
assert.throws(() => toml.loads('big = 9223372036854775808'), /toml/i)

let deep = { value: 1 }
for (let index = 0; index < 49; index += 1) deep = { nested: deep }
assert.throws(() => toml.dumps(deep), /depth/i)

## Placeholders

In [ ]:
const assert = require('node:assert/strict')
const { toml } = require('yggdryl')

const document = '[database]\nhost = "{{ HOST }}"\nport = "{{ PORT }}"\n'
const value = toml.loads(document, {
  placeholders: { HOST: 'db.internal', PORT: 5432 },
})
assert.deepEqual(value.database, { host: 'db.internal', port: 5432 })